In [1]:
%matplotlib inline
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd


from dotenv import load_dotenv
from sqlalchemy import create_engine

_NOTEBOOK_DIR = Path.cwd()
_PROJECT_ROOT = _NOTEBOOK_DIR
for _ in range(5):
    if (_PROJECT_ROOT / ".env").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent

# Ensure Training dir is on path so data_prep and data_prep_batters can be imported
sys.path.insert(0, str(_NOTEBOOK_DIR))

def _load_env():
    load_dotenv(_PROJECT_ROOT / ".env")

def _db_url():
    host = os.getenv("PGHOST")
    port = os.getenv("PGPORT")
    user = os.getenv("PGUSER")
    password = os.getenv("PGPASSWORD")
    dbname = os.getenv("PGDATABASE")
    pw = quote_plus(password) if password else ""
    return f"postgresql://{user}:{pw}@{host}:{port}/{dbname}"

_load_env()
engine = create_engine(_db_url())


In [2]:
from sklearn.metrics import log_loss, accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split

import numpy as np
import optuna
import xgboost as xgb

from data_prep import load_pitcher_simulator_data, prepare_pitcher_features

# Deciding counts (3-2, 0-2, 1-2, 2-2, 3-1) for count-aware sample weighting
DECIDING_COUNTS = {(0, 2), (1, 2), (2, 2), (3, 1), (3, 2)}
def is_deciding_count(balls: int, strikes: int) -> bool:
    return (int(balls), int(strikes)) in DECIDING_COUNTS

print("imported modules")

imported modules


c:\Users\macia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_raw = load_pitcher_simulator_data(engine)
X, y_dict = prepare_pitcher_features(df_raw)

print(f"Loaded {len(X)} pitches")
print(f"Features: {list(X.columns)}")
print(f"Targets: {list(y_dict.keys())}")

X_train, X_test, y_train_dict, y_test_dict = {}, {}, {}, {}
train_idx, test_idx = train_test_split(
    np.arange(len(X)),
    test_size=0.2,
    random_state=42,
    stratify=y_dict["pitch_type"],
)
X_train = X.iloc[train_idx].copy().reset_index(drop=True)
X_test = X.iloc[test_idx].copy().reset_index(drop=True)
for t in y_dict:
    y_train_dict[t] = y_dict[t].iloc[train_idx].reset_index(drop=True)
    y_test_dict[t] = y_dict[t].iloc[test_idx].reset_index(drop=True)

print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

Loaded 493360 pitches
Features: ['pitcher', 'p_throws', 'stand', 'balls', 'strikes', 'is_pitcher_count', 'is_batter_count', 'inning', 'inning_topbot', 'outs_when_up', 'at_bat_number', 'pitch_number', 'previous_pitch_type', 'previous_release_speed', 'pitcher_pitches_this_game', 'pitcher_pitches_this_inning', 'game_date', 'home_team', 'away_team', 'game_type', 'pitcher_career_ip', 'pitcher_career_era', 'pitcher_career_so', 'pitcher_career_bb', 'pitcher_career_h', 'pitcher_career_er', 'pitcher_career_hr', 'pitcher_career_bfp', 'pitcher_career_ipouts']
Targets: ['pitch_type', 'plate_x', 'plate_z', 'release_speed', 'release_spin_rate']

Train: 394688 | Test: 98672


In [4]:
import json

repertoire_by_pitcher = (
    pd.DataFrame({"pitcher": X_train["pitcher"], "pitch_type": y_train_dict["pitch_type"]})
    .groupby("pitcher")["pitch_type"]
    .apply(lambda s: sorted(s.unique().tolist()))
    .to_dict()
)
# JSON keys must be strings
pitcher_repertoire_json = {str(int(k)): v for k, v in repertoire_by_pitcher.items()}
_SAVED = Path(_NOTEBOOK_DIR) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
with open(_SAVED / "pitcher_repertoire.json", "w") as f:
    json.dump(pitcher_repertoire_json, f, indent=2)
print(f"Saved pitcher_repertoire.json ({len(pitcher_repertoire_json)} pitchers)")

Saved pitcher_repertoire.json (872 pitchers)


In [5]:
from sklearn.utils.class_weight import compute_class_weight

codes_in_train = sorted(y_train_dict["pitch_type"].unique())
type_to_idx = {t: i for i, t in enumerate(codes_in_train)}
map_pitch_types = {i: t for t, i in type_to_idx.items()}

y_train_pt = y_train_dict["pitch_type"].map(type_to_idx)
y_test_pt = y_test_dict["pitch_type"].map(lambda t: type_to_idx.get(t, 0))

classes_pt = np.unique(y_train_pt)
class_weights_pt = compute_class_weight(
    "balanced", classes=classes_pt, y=y_train_pt.to_numpy().ravel()
)
class_weight_per_sample = class_weights_pt[np.searchsorted(classes_pt, y_train_pt.to_numpy().ravel())]

# Count-awareness: upweight deciding counts (3-2, 0-2, 1-2, 2-2, 3-1) so model learns pitcher behavior in those counts
DECIDING_COUNT_WEIGHT = 1.5
is_deciding_train = X_train.apply(lambda r: is_deciding_count(int(r["balls"]), int(r["strikes"])), axis=1)
deciding_multiplier = np.where(is_deciding_train.values, DECIDING_COUNT_WEIGHT, 1.0)
sample_weight_pt = class_weight_per_sample * deciding_multiplier

# For regressors (same row order as X_train)
sample_weight_reg = deciding_multiplier.astype(np.float64)

X_train_pt = X_train
y_train_pt_oversampled = y_train_pt

feats_s1 = list(X_train.columns)

In [6]:
# Hyperparameter tuning: Optuna runs many trials (random sampling of n_estimators, max_depth,
# learning_rate) and we keep the config that minimizes log loss on the test set
def tune_pitch_type(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 500)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    model = xgb.XGBClassifier(
        objective="multi:softprob", random_state=42,
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=learning_rate, subsample=subsample, colsample_bytree=colsample_bytree,
    )
    model.fit(X_train_pt[feats_s1], y_train_pt_oversampled, sample_weight=sample_weight_pt)
    proba = model.predict_proba(X_test[feats_s1])
    # Pass labels so log_loss accepts when test set has fewer classes than train (e.g. 16 vs 17)
    return log_loss(y_test_pt, proba, labels=np.arange(len(codes_in_train)))

study_pt = optuna.create_study(direction="minimize")
study_pt.optimize(tune_pitch_type, n_trials=10)
print("Best log_loss:", study_pt.best_value, "| Best params:", study_pt.best_params)


[I 2026-03-05 00:49:50,638] A new study created in memory with name: no-name-8e931850-4696-4945-829a-1a9082ee5118
[I 2026-03-05 00:50:43,000] Trial 0 finished with value: 1.3976031468579106 and parameters: {'n_estimators': 334, 'max_depth': 8, 'learning_rate': 0.12170009589584578, 'subsample': 0.5429522892087552, 'colsample_bytree': 0.8671988461586042}. Best is trial 0 with value: 1.3976031468579106.
[I 2026-03-05 00:51:23,987] Trial 1 finished with value: 1.5072480950621865 and parameters: {'n_estimators': 330, 'max_depth': 4, 'learning_rate': 0.2703808570002039, 'subsample': 0.5764220529823416, 'colsample_bytree': 0.5186679577702231}. Best is trial 0 with value: 1.3976031468579106.
[I 2026-03-05 00:52:09,975] Trial 2 finished with value: 1.567895402250928 and parameters: {'n_estimators': 375, 'max_depth': 3, 'learning_rate': 0.2599539425687737, 'subsample': 0.5708616886333895, 'colsample_bytree': 0.8153650630844853}. Best is trial 0 with value: 1.3976031468579106.
[I 2026-03-05 00:52

Best log_loss: 1.352804948627055 | Best params: {'n_estimators': 474, 'max_depth': 12, 'learning_rate': 0.04071595242918971, 'subsample': 0.6692322450409935, 'colsample_bytree': 0.6940291682867799}


In [7]:
# Addressing memorizing the majority / overfitting: we oversample rare pitch types (and cap
# common ones at the median count) so the model learns all types instead of memorizing noise
# or collapsing to the most frequent class
rs = np.random.RandomState(42)
n_classes = len(codes_in_train)
counts = y_train_pt.value_counts().reindex(range(n_classes), fill_value=0).values
target_per_class = int(np.median(counts[counts > 0]))  # avoid huge dataset
balanced_idx = []
for c in range(n_classes):
    idx_c = np.where(y_train_pt.values == c)[0]
    n_c = len(idx_c)
    if n_c == 0:
        continue
    if n_c >= target_per_class:
        chosen = rs.choice(idx_c, size=target_per_class, replace=False)
    else:
        chosen = rs.choice(idx_c, size=target_per_class, replace=True)
    balanced_idx.extend(chosen)
balanced_idx = np.array(balanced_idx)
rs.shuffle(balanced_idx)

X_train_pt = X_train.iloc[balanced_idx].reset_index(drop=True)
y_train_pt_oversampled = y_train_pt.iloc[balanced_idx].reset_index(drop=True)
# Align sample weights with oversampled rows (same length as X_train_pt)
sample_weight_pt = np.asarray(sample_weight_pt)[balanced_idx]
print(f"Pitch-type training: original {len(X_train)} -> oversampled {len(X_train_pt)} (target {target_per_class} per class)")

Pitch-type training: original 394688 -> oversampled 114478 (target 6734 per class)


In [8]:
xgb_pt = xgb.XGBClassifier(objective="multi:softprob", random_state=42, **study_pt.best_params)
xgb_pt.fit(X_train_pt[feats_s1], y_train_pt_oversampled, sample_weight=sample_weight_pt)

col_names = [map_pitch_types[i] for i in range(len(codes_in_train))]
proba_pt = xgb_pt.predict_proba(X_test[feats_s1])
pitch_type_pred = pd.Series(
    [col_names[i] for i in np.argmax(proba_pt, axis=1)],
    index=X_test.index,
)

_SAVED = Path(_NOTEBOOK_DIR) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
xgb_pt.save_model(str(_SAVED / "pitcher_pitch_type.json"))
print("Saved pitcher_pitch_type.json")

Saved pitcher_pitch_type.json


In [9]:
X_train_s2 = X_train.copy()
X_train_s2["pitch_type_code"] = y_train_dict["pitch_type"].map(type_to_idx)

X_test_s2 = X_test.copy()
X_test_s2["pitch_type_code"] = pitch_type_pred.map(lambda t: type_to_idx.get(t, 0))

feats_s2 = feats_s1 + ["pitch_type_code"]

In [10]:
# Hyperparameter tuning for the four regressors (plate_x, plate_z, release_speed, release_spin_rate)
# Optuna runs n_trials per target and we fit and save the model with the best params for each
def tune_reg(trial, target_name):
    n_estimators = trial.suggest_int("n_estimators", 50, 500)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 0.01, 10.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.1, 10.0)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42,
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=learning_rate, subsample=subsample, colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha, reg_lambda=reg_lambda, min_child_weight=min_child_weight)
    model.fit(X_train_s2[feats_s2], y_train_dict[target_name], sample_weight=sample_weight_reg)
    pred = model.predict(X_test_s2[feats_s2])
    return np.sqrt(mean_squared_error(y_test_dict[target_name], pred))  # RMSE

reg_targets = ["plate_x", "plate_z", "release_speed", "release_spin_rate"]
reg_models = {}
for tgt in reg_targets:
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial, t=tgt: tune_reg(trial, t), n_trials=10)
    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42, **study.best_params)
    model.fit(X_train_s2[feats_s2], y_train_dict[tgt], sample_weight=sample_weight_reg)
    model.save_model(str(_SAVED / f"pitcher_{tgt}.json"))
    reg_models[tgt] = model
    print(f"pitcher_{tgt}.json: best RMSE = {study.best_value:.4f}")

[I 2026-03-05 00:59:31,154] A new study created in memory with name: no-name-a85f8c07-62f4-4c73-9f67-682c97158f13
[I 2026-03-05 00:59:38,739] Trial 0 finished with value: 0.8771505484415311 and parameters: {'n_estimators': 486, 'max_depth': 10, 'learning_rate': 0.29151201274909594, 'subsample': 0.6700478840961561, 'colsample_bytree': 0.9727330345591254, 'reg_alpha': 4.913965498267702, 'reg_lambda': 3.0142579224744876, 'min_child_weight': 10}. Best is trial 0 with value: 0.8771505484415311.
[I 2026-03-05 00:59:40,701] Trial 1 finished with value: 0.826356403739892 and parameters: {'n_estimators': 193, 'max_depth': 8, 'learning_rate': 0.29248873600546893, 'subsample': 0.9422757388508212, 'colsample_bytree': 0.8990918850907295, 'reg_alpha': 8.075112955967885, 'reg_lambda': 4.667995755568727, 'min_child_weight': 1}. Best is trial 1 with value: 0.826356403739892.
[I 2026-03-05 00:59:44,465] Trial 2 finished with value: 0.8523883510214412 and parameters: {'n_estimators': 379, 'max_depth': 8,

pitcher_plate_x.json: best RMSE = 0.8129


[I 2026-03-05 01:00:13,283] Trial 0 finished with value: 1.0547603863825639 and parameters: {'n_estimators': 458, 'max_depth': 10, 'learning_rate': 0.15495524859816978, 'subsample': 0.7565323613265502, 'colsample_bytree': 0.9066022977287926, 'reg_alpha': 8.743131129466253, 'reg_lambda': 9.143809836670691, 'min_child_weight': 3}. Best is trial 0 with value: 1.0547603863825639.
[I 2026-03-05 01:00:21,331] Trial 1 finished with value: 1.0407503471838802 and parameters: {'n_estimators': 410, 'max_depth': 12, 'learning_rate': 0.04584030489319478, 'subsample': 0.7911859167718668, 'colsample_bytree': 0.7457358506632417, 'reg_alpha': 6.707209006054512, 'reg_lambda': 7.1410898013065465, 'min_child_weight': 10}. Best is trial 1 with value: 1.0407503471838802.
[I 2026-03-05 01:00:22,401] Trial 2 finished with value: 1.0352068513837283 and parameters: {'n_estimators': 187, 'max_depth': 5, 'learning_rate': 0.21048403077823527, 'subsample': 0.9968156698563162, 'colsample_bytree': 0.9209996363067215,

pitcher_plate_z.json: best RMSE = 1.0327


[I 2026-03-05 01:00:51,442] Trial 0 finished with value: 7.620509335572973 and parameters: {'n_estimators': 406, 'max_depth': 10, 'learning_rate': 0.2417618402691346, 'subsample': 0.9199945122295927, 'colsample_bytree': 0.6784956684908594, 'reg_alpha': 9.089892096077508, 'reg_lambda': 2.43886581644776, 'min_child_weight': 2}. Best is trial 0 with value: 7.620509335572973.
[I 2026-03-05 01:00:52,285] Trial 1 finished with value: 7.540322563852214 and parameters: {'n_estimators': 94, 'max_depth': 7, 'learning_rate': 0.18080827810596795, 'subsample': 0.6995904714950831, 'colsample_bytree': 0.6416766487573867, 'reg_alpha': 4.069790628215908, 'reg_lambda': 0.4317563875205209, 'min_child_weight': 9}. Best is trial 1 with value: 7.540322563852214.
[I 2026-03-05 01:00:54,642] Trial 2 finished with value: 7.61093324664735 and parameters: {'n_estimators': 342, 'max_depth': 7, 'learning_rate': 0.18004338638594924, 'subsample': 0.8838027488010416, 'colsample_bytree': 0.9767922662064159, 'reg_alpha

pitcher_release_speed.json: best RMSE = 7.3534


[I 2026-03-05 01:01:15,617] Trial 0 finished with value: 443.6157341858257 and parameters: {'n_estimators': 283, 'max_depth': 6, 'learning_rate': 0.15990080182147232, 'subsample': 0.9616418069178408, 'colsample_bytree': 0.5959262612319558, 'reg_alpha': 5.300225678477622, 'reg_lambda': 8.886282350322368, 'min_child_weight': 3}. Best is trial 0 with value: 443.6157341858257.
[I 2026-03-05 01:01:18,689] Trial 1 finished with value: 450.7541849369123 and parameters: {'n_estimators': 373, 'max_depth': 7, 'learning_rate': 0.23437839586715395, 'subsample': 0.6929943641429026, 'colsample_bytree': 0.8929810564755973, 'reg_alpha': 0.9824799946198833, 'reg_lambda': 9.628935612636234, 'min_child_weight': 8}. Best is trial 0 with value: 443.6157341858257.
[I 2026-03-05 01:01:21,564] Trial 2 finished with value: 452.22730372703177 and parameters: {'n_estimators': 302, 'max_depth': 9, 'learning_rate': 0.28858717061667816, 'subsample': 0.7852300394201162, 'colsample_bytree': 0.5718877769100779, 'reg_a

pitcher_release_spin_rate.json: best RMSE = 440.8250


In [11]:
#compute how well the pitch type model and the four regressors do on the test set
acc_pt = accuracy_score(y_test_pt, xgb_pt.predict(X_test[feats_s1]))
proba_pt_test = xgb_pt.predict_proba(X_test[feats_s1])
y_test_onehot = np.zeros_like(proba_pt_test)
for i, pt in enumerate(y_test_pt):
    if 0 <= pt < proba_pt_test.shape[1]:
        y_test_onehot[i, int(pt)] = 1
loss_pt = log_loss(y_test_onehot, proba_pt_test)
print("Pitch type: accuracy =", f"{acc_pt:.4f}", "| log loss =", f"{loss_pt:.4f}")

for tgt in reg_targets:
    pred = reg_models[tgt].predict(X_test_s2[feats_s2])
    rmse = np.sqrt(mean_squared_error(y_test_dict[tgt], pred))
    mae = np.abs(y_test_dict[tgt].values - pred).mean()
    print(f"{tgt}: RMSE = {rmse:.4f} | MAE = {mae:.4f}")

Pitch type: accuracy = 0.3153 | log loss = 2.3095
plate_x: RMSE = 0.8129 | MAE = 0.6427
plate_z: RMSE = 1.0327 | MAE = 0.8098
release_speed: RMSE = 7.3534 | MAE = 5.8254
release_spin_rate: RMSE = 440.8250 | MAE = 304.9497


In [12]:
# Monte Carlo at inference + avoiding memorizing noise: compute the std of prediction errors
# per pitch type and save it. The app then adds random noise with this scale when predicting,
# so the simulator outputs a distribution of plausible outcomes instead of a single deterministic
# prediction that would overfit to training noise
import json

pt_train = y_train_dict["pitch_type"]  # pitch type strings
residual_stds = {}
DEFAULT_STDS = {"plate_x": 0.5, "plate_z": 0.5, "release_speed": 1.5, "release_spin_rate": 200}
MIN_SAMPLES = 10

for tgt in reg_targets:
    pred = reg_models[tgt].predict(X_train_s2[feats_s2])
    res = y_train_dict[tgt].values - pred
    df_res = pd.DataFrame({"pt": pt_train.values, "res": res})
    by_pt = df_res.groupby("pt")["res"].agg(["std", "count"])
    global_std = df_res["res"].std()
    std_by_pt = {}
    for pt in by_pt.index:
        row = by_pt.loc[pt]
        std_by_pt[pt] = float(row["std"]) if row["count"] >= MIN_SAMPLES and pd.notna(row["std"]) else float(global_std)
    residual_stds[tgt] = std_by_pt

with open(_SAVED / "residual_stds.json", "w") as f:
    json.dump(residual_stds, f, indent=2)
print("Saved residual_stds.json")

Saved residual_stds.json


In [13]:
#for each pitcher and each pitch type compute average location and save so the app can nudge predictions toward it
MIN_PITCHES_FOR_MEANS = 20
df_loc = pd.DataFrame({
    "pitcher": X_train["pitcher"],
    "pitch_type": y_train_dict["pitch_type"],
    "plate_x": y_train_dict["plate_x"],
    "plate_z": y_train_dict["plate_z"],
})
by_pitcher_pt = df_loc.groupby(["pitcher", "pitch_type"]).agg(
    plate_x=("plate_x", "mean"),
    plate_z=("plate_z", "mean"),
    n=("plate_x", "count"),
).reset_index()
pitcher_plate_means = {}
for _, row in by_pitcher_pt.iterrows():
    if row["n"] < MIN_PITCHES_FOR_MEANS:
        continue
    pid = str(int(row["pitcher"]))
    pt = row["pitch_type"]
    if pid not in pitcher_plate_means:
        pitcher_plate_means[pid] = {}
    pitcher_plate_means[pid][pt] = {"plate_x": float(row["plate_x"]), "plate_z": float(row["plate_z"])}
with open(_SAVED / "pitcher_plate_means.json", "w") as f:
    json.dump(pitcher_plate_means, f, indent=2)
print(f"Saved pitcher_plate_means.json ({len(pitcher_plate_means)} pitchers with >= {MIN_PITCHES_FOR_MEANS} pitches per type)")

Saved pitcher_plate_means.json (756 pitchers with >= 20 pitches per type)


In [14]:
#list the model and json files we wrote to saved_models
list(_SAVED.glob("pitcher_*.json"))

[WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_pitch_type.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_pitch_type_rates.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_means.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_x.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_z.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_release_speed.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_release_spin_rate.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/

In [15]:
#for each pitcher compute how often they threw each pitch type and save so the app can blend with model probs
import json
df_pt = pd.DataFrame({"pitcher": X_train["pitcher"], "pitch_type": y_train_dict["pitch_type"]})
rates = df_pt.groupby("pitcher")["pitch_type"].value_counts(normalize=True).unstack(fill_value=0.0)
pitcher_pitch_type_rates = {}
for pid in rates.index:
    d = rates.loc[pid].to_dict()
    pitcher_pitch_type_rates[str(int(pid))] = {str(k): float(v) for k, v in d.items()}
with open(_SAVED / "pitcher_pitch_type_rates.json", "w") as f:
    json.dump(pitcher_pitch_type_rates, f, indent=2)
print(f"Saved pitcher_pitch_type_rates.json ({len(pitcher_pitch_type_rates)} pitchers)")

Saved pitcher_pitch_type_rates.json (872 pitchers)
